# BTE 413 Final Group Project: predicting the cost of a hospital stay at admission
**Regression group demo.** Run **Runtime > Run all**. It takes about a minute and needs no Google Drive: the data loads from a public link.

What it shows, in order: (1) the data, (2) which variables help, (3) the final model and its test score, (4) what drives cost. A check at the end compares every number with the slides.

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, median_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import OneHotEncoder

# Public link to the cleaned data (NY SPARCS Hospital Inpatient Discharges 2024, de-identified, 200,000 random stays)
DATA_URL = 'https://raw.githubusercontent.com/ethanmooseg-star/bte413-hospital-cost/main/sparcs_2024_clean_sample.csv.gz'

## 1. The data
Cleaned by `clean_sparcs_2024.py` (in the same repository). The train and test labels are fixed in the `split` column, so everyone uses the same rows.

In [ ]:
df = pd.read_csv(DATA_URL, dtype={'permanent_facility_id': str, 'apr_mdc_code': str, 'apr_drg_code': str,
                                  'zip_code': str, 'ccsr_diagnosis_code': str, 'ccsr_procedure_code': str})
train, test = df[df['split'] == 'train'], df[df['split'] == 'test']
print(f"{len(df):,} stays | {len(train):,} train | {len(test):,} test")
print(f"Cost is very skewed: average ${df['total_costs'].mean():,.0f}, middle ${df['total_costs'].median():,.0f}, so we predict the log of cost.")

## 2. Which variables help?
**The rule:** the model runs at admission, so it may only use what a hospital knows when the patient arrives. Diagnosis, procedure, severity, length of stay and discharge status are set later, so they are not allowed.

A Fisher-style F test on the train rows: how much of the difference in cost each admission-time variable explains.

In [ ]:
def f_test(t, col):
    y = t['log_total_costs']; g = y.groupby(t[col])
    ssb = (g.size() * (g.mean() - y.mean()) ** 2).sum()
    k, n = g.ngroups, len(t); sst = ((y - y.mean()) ** 2).sum()
    return (ssb / (k - 1)) / ((sst - ssb) / (n - k)), ssb / sst

ADMISSION = ['age_group', 'type_of_admission', 'emergency_department_indicator', 'payment_typology_1',
             'health_service_area', 'permanent_facility_id', 'hospital_county', 'gender', 'race', 'ethnicity']
ranking = pd.DataFrame([(c, *f_test(train, c)) for c in ADMISSION], columns=['variable', 'F', 'share_of_cost_variance'])
print(ranking.sort_values('share_of_cost_variance', ascending=False).round(3).to_string(index=False))
print("\nFacility's F is low only because it has 204 groups; compare the share of variance explained.")

## 3. The final model and its test score
Linear regression on four variables: age group, type of admission, main payer, facility. It is explainable, which a hospital needs.

We train and test on the 90% of stays under the high-cost cutoff. The cutoff uses the final cost, so a hospital cannot apply it at admission: this model describes typical stays.

In [ ]:
LEAN = ['age_group', 'type_of_admission', 'payment_typology_1', 'permanent_facility_id']
cutoff = df.loc[df['high_cost'] == 1, 'total_costs'].min()
trim_train, trim_test = train[train['high_cost'] == 0], test[test['high_cost'] == 0]
print(f"Cutoff ${int(cutoff):,}: {len(trim_train):,} train and {len(trim_test):,} test stays remain")

enc = OneHotEncoder(handle_unknown='ignore').fit(trim_train[LEAN])
final = LinearRegression().fit(enc.transform(trim_train[LEAN]), trim_train['log_total_costs'])

def report(name, t):
    p = final.predict(enc.transform(t[LEAN])); e = np.abs(t['total_costs'] - np.exp(p))
    r2, typical, avg = r2_score(t['log_total_costs'], p), median_absolute_error(t['total_costs'], np.exp(p)), mean_absolute_error(t['total_costs'], np.exp(p))
    print(f"{name} ({len(t):,}): R2 (log) = {r2:.3f} | typical miss = ${typical:,.0f} | average miss = ${avg:,.0f}")
    return r2, typical, avg

typical_stays = report('FINAL model on typical test stays', trim_test)
all_stays = report('Same model on ALL test stays      ', test)

## 4. What drives cost
Percent difference in cost against a patient who is 70 or older, admitted through the emergency department, with Medicare. Negative means cheaper.

In [ ]:
REF = {'age_group': '70 or Older', 'type_of_admission': 'Emergency', 'payment_typology_1': 'Medicare'}
Xd = pd.get_dummies(trim_train[LEAN]).astype(float)
lm = LinearRegression().fit(Xd, trim_train['log_total_costs'])
co = pd.Series(lm.coef_, index=Xd.columns)
rows = [(v, c[len(v) + 1:], round((np.exp(co[c] - co[v + '_' + REF[v]]) - 1) * 100, 1))
        for v in REF for c in co.index if c.startswith(v + '_') and c != v + '_' + REF[v]]
print(pd.DataFrame(rows, columns=['variable', 'group', 'cost_change_pct']).sort_values(['variable', 'cost_change_pct']).to_string(index=False))
common = trim_train['permanent_facility_id'].value_counts(); base = common.index[0]
fac = pd.Series({c[len('permanent_facility_id_'):]: (np.exp(co[c] - co['permanent_facility_id_' + base]) - 1) * 100
                 for c in co.index if c.startswith('permanent_facility_id_')})
fac = fac[[i for i in fac.index if common.get(i, 0) >= 300]].sort_values()
print(f"\nHospitals against the most common one ({base}): 3 lowest and 3 highest")
print(pd.concat([fac.head(3), fac.tail(3)]).round(1).to_string())

## 5. Do the numbers match the slides?

In [ ]:
checks = [
    ('Final model R2, typical stays', typical_stays[0], 0.393, 0.001),
    ('Final model typical miss ($)', typical_stays[1], 5005, 5),
    ('Final model average miss ($)', typical_stays[2], 7571, 5),
    ('Same model R2, all test stays', all_stays[0], 0.312, 0.001),
    ('Same model average miss ($)', all_stays[2], 17148, 5),
]
for name, got, want, tol in checks:
    print(f"{'OK     ' if abs(got - want) <= tol else 'DIFFERS'}  {name}: {got:,.3f} (slides: {want:,})")